# 3. Data Modeling & Training (Tokenized Dataset) - Akkadian to English Translation 
## Aaron Dichoso & Luis Razon

This notebook details the steps performed for training a model on the tokenized dataset used in the Deep Past Challenge for Translating Akkadian Text to English.
The competition can be accessed in this link: https://www.kaggle.com/competitions/deep-past-initiative-machine-translation/data

Run this notebook AFTER running "2. Preprocessing".

In [ ]:
import pandas as pd
import numpy
print(numpy.__version__)
print(pd.__version__)
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

processed_complete_df = pd.read_csv("processed/processed_train_complete.csv")
processed_incomplete_df = pd.read_csv("processed/processed_train_incomplete.csv")

processed_complete_df.sample(5)

In [ ]:
from ast import literal_eval

#Convert data type of columns from strings to array objects
processed_complete_df["transliteration"] = processed_complete_df["transliteration"].apply(literal_eval)
processed_incomplete_df["transliteration"] = processed_incomplete_df["transliteration"].apply(literal_eval)
processed_complete_df["translation"] = processed_complete_df["translation"].apply(literal_eval)
processed_incomplete_df["translation"] = processed_incomplete_df["translation"].apply(literal_eval)

The Data Modeling Pipeline is as follows:

Tokens -> Token IDs -> Model Training (w/ Embeddings) -> Token Decoding

First, we need to convert each token into a set of token IDs for the model to learn.

In [ ]:
import utils.bpe as bpe

#Import BPE vocab and tokens
BPE = bpe.BytePairEncoder()
BPE.load("processed/akk2eng.json")

In [ ]:
vocab = {}

#Get all tokens in the BPE
for i, token in enumerate(sorted(BPE.tokens.keys())):
    vocab[token] = i

#Ensure the sos and eos tokens exist 
#<pad> token for padding in transformer because of different sizes of input, <unk> token for unknown predictions by model, - for Akkadian
specials = ['<pad>', '<unk>', '<sos>', '-', '<eos>']

for tok in specials:
    if tok not in vocab:
        vocab[tok] = len(vocab)

len(vocab), vocab["<eos>"]

In [ ]:
#Function used to convert tokens to token ids
def tokens_to_ids(tokens):
    return [vocab.get(t, vocab['<unk>']) for t in tokens]

In [ ]:
import torch
from torch.utils.data import Dataset

#Class used for the dataset used in training
class TranslationDataset(Dataset):
    def __init__(self, dataframe, src_row="transliteration", tgt_row="translation", pad_id=vocab['<pad>']):
        self.df = dataframe.reset_index(drop=True)
        self.src_row = src_row
        self.tgt_row = tgt_row

        self.process_dataset()
        self.pad_id = pad_id

    def process_dataset(self):
        #Tokens to IDs
        for i, row in self.df.iterrows():
            self.df.at[i, self.src_row] = tokens_to_ids(row[self.src_row])
            self.df.at[i, self.tgt_row] = tokens_to_ids(row[self.tgt_row])
        
        return
    
    def __len__(self):
        return len(self.df)

    #Entry to tensor
    def __getitem__(self, idx):
        src = self.df.loc[idx, self.src_row]
        tgt = self.df.loc[idx, self.tgt_row]

        # convert to tensor
        src = torch.tensor(src, dtype=torch.long)
        tgt = torch.tensor(tgt, dtype=torch.long)

        return src, tgt

In [ ]:
# Take note of the max length that input and output text can be
MAX_LEN = max(max(len(x) for x in processed_complete_df["transliteration"]), 
              max(len(x) for x in processed_complete_df["translation"]))

MAX_LEN

In [ ]:
#MAX_LEN=384

## Dataset Definition
This section deals with defining the dataset to be used in model training

In [ ]:
import torch
import torch.optim as optim
from torch.utils.data import DataLoader, random_split

In [ ]:
from torch.nn.utils.rnn import pad_sequence
#device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = "cpu"
#Used by dataloader so that it knows how to compile batches together
def collate_fn(batch):
    src_batch, tgt_batch = zip(*batch)

    #Pad all sequences so that they are of equal length
    src_batch = pad_sequence(
        src_batch,
        batch_first=True,
        padding_value=vocab["<pad>"]
    ).to(device)

    tgt_batch = pad_sequence(
        tgt_batch,
        batch_first=True,
        padding_value=vocab["<pad>"]
    ).to(device)

    return src_batch, tgt_batch

In [ ]:
# Dataset
dataset = TranslationDataset(processed_complete_df)

# Split into train / validation
val_ratio = 0.1  # 10% for validation
val_size = int(len(dataset) * val_ratio)
train_size = len(dataset) - val_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

# DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn
)
print(f"Train samples: {len(train_dataset)}, Validation samples: {len(val_dataset)}")

## Translator
This sections deals with the decoder used to reconstruct the tokens from the model. Viewing other model submissions from the competition shows the Minimum Bayes Risk (MBR) decoding method to be the way to go. We use this source as reference for our implementation (https://suzyahyah.github.io/bayesian%20inference/machine%20translation/2022/02/15/mbr-decoding.html).

MBR Decoding applied in this case reduces to consensus decoding, which means that we can just run multiple predictions from our model and pick the prediction most similar to the other predictions.

In [ ]:
#Dictionary for translating IDs back to tokens.
inv_vocab = {v: k for k, v in vocab.items()}

In [ ]:
def decode_tokens(token_ids):
    tokens = []
    #Translate through the dictionary
    for idx in token_ids:
        token = inv_vocab.get(idx, "<unk>")
        if token in ["<pad>", "<sos>", "<eos>"]: #Skip when you see these characters.
            continue
        tokens.append(token)
    return " ".join(tokens)

In [ ]:
def sample_decode_batched_l(model, src, max_len, sos_id, eos_id, samples=8):
    model.eval()
    with torch.no_grad():
        # Duplicate src set [sample] number of times for batch decoding (samples, src_len).
        src_expanded = src.repeat(samples, 1)
        
        #Encoder pass for sample batch
        encoder_outputs, hidden, cell = model.encoder(src_expanded)
        
        #Initialized decoded text array (starts with <sos>)
        generated = torch.full(
            (samples, 1), sos_id, dtype=torch.long, device=src.device
        )

        #Monitor finished sequences to stop decoding early
        finished = torch.zeros(samples, dtype=torch.bool, device=src.device)

        #Predict each next token
        for _ in range(max_len):
            logits, hidden, cell = model.decoder(generated, hidden, cell, encoder_outputs)
            
            # Sample from the last token's distribution
            probs = logits[:, -1].softmax(-1)
            next_tokens = probs.multinomial(1)  # (samples, 1)
            next_tokens[finished] = eos_id

            generated = torch.cat([generated, next_tokens], dim=1)
            
            # Mark newly finished samples
            finished |= (next_tokens.squeeze(1) == eos_id)

            if finished.all():
                break

    return generated[:, 1:]  # remove <sos>

In [ ]:
def sample_decode_batched_t(model, src, max_len, sos_id, eos_id, samples=8):
    model.eval()
    with torch.no_grad():
        # Duplicate src for batch decoding (samples, src_len)
        src_expanded = src.repeat(samples, 1)

        # Initialized decoded text array (starts with <sos>)
        generated = torch.full(
            (samples, 1), sos_id, dtype=torch.long, device=src.device
        )

        # Monitor finished sequences to stop decoding early
        finished = torch.zeros(samples, dtype=torch.bool, device=src.device)

        for _ in range(max_len):
            # transformer takes full src and full generated sequence so far
            logits = model(src_expanded, generated)

            # sample from the last token's distribution
            probs = logits[:, -1].softmax(-1)
            next_tokens = probs.multinomial(1)  # (samples, 1)
            next_tokens[finished.unsqueeze(1)] = eos_id

            generated = torch.cat([generated, next_tokens], dim=1)

            # mark newly finished samples
            finished |= (next_tokens.squeeze(1) == eos_id)

            if finished.all():
                break

    return generated[:, 1:]  # remove <sos>

#### Similarity Score
The similarity score we will use is the geometric mean between BLEU and chrF++, as this is the same metric used in the Kaggle competition.

In [ ]:
from sacrebleu.metrics import BLEU, CHRF
import numpy as np

bleu = BLEU(effective_order=True)
chrf = CHRF(word_order=2) 

In [ ]:
def detokenize(tokens):
    # join tokens and strip the BPE end-of-word marker. Needed as SacreBLEU requires plain text as input.
    return ' '.join(tokens).replace('_ ', ' ').replace('_', '').strip()

In [ ]:
def similarity(hypothesis_tokens, reference_tokens):
    hyp = detokenize(decode_tokens(hypothesis_tokens))
    ref = detokenize(decode_tokens(reference_tokens))
    bleu_score = bleu.sentence_score(hyp, [ref]).score / 100
    chrf_score = chrf.sentence_score(hyp, [ref]).score / 100
    return np.sqrt(bleu_score * chrf_score)

In [ ]:
def truncate_at_eos(tokens, eos_id):
    if eos_id in tokens:
        return tokens[:tokens.index(eos_id)]
    return tokens

Below is the function that performs MBR decoding for us.

In [ ]:
def mbr_decode(model, src, max_len, sos_id, eos_id, model_type="l", samples=8):
    #Get candidate translations from the model

    if model_type == "l":
        all_samples = sample_decode_batched_l(model, src, max_len, sos_id, eos_id, samples)
    elif model_type == "t":
        all_samples = sample_decode_batched_t(model, src, max_len, sos_id, eos_id, samples)
    
    candidates = all_samples.cpu().tolist()

    # Truncate each candidate at <eos> before scoring
    candidates = [truncate_at_eos(c, eos_id) for c in candidates]
    scores = []
    for i, c1 in enumerate(candidates):
        score = sum(similarity(c1, c2) for j, c2 in enumerate(candidates) if i != j)
        scores.append(score)

    return candidates[scores.index(max(scores))]

# Approach 1. Transformers

In [ ]:
import torch
import torch.nn as nn

class Seq2SeqTransformer(nn.Module):
    def __init__(self, vocab_size, pad_id, max_len=MAX_LEN):
        super().__init__()

        self.d_model = 128
        self.pad_id = pad_id

        self.token_embedding = nn.Embedding(vocab_size, self.d_model, padding_idx=vocab['<pad>'])
        self.pos_embedding = nn.Embedding(max_len, self.d_model)

        self.transformer = nn.Transformer(
            d_model=self.d_model,
            nhead=2,
            num_encoder_layers=2,
            num_decoder_layers=2,
            dim_feedforward=256,
            dropout=0.2,
            batch_first=True
        )

        self.fc_out = nn.Linear(self.d_model, vocab_size)

    def make_src_mask(self, src):
        return (src == self.pad_id)

    def make_tgt_mask(self, tgt):
        seq_len = tgt.size(1)
        mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
        return mask.to(tgt.device)

    def forward(self, src, tgt):
        # src, tgt are token IDs
        src_positions = torch.arange(0, src.size(1), device=src.device).unsqueeze(0)
        tgt_positions = torch.arange(0, tgt.size(1), device=tgt.device).unsqueeze(0)

        # Token + Positional embeddings
        src_emb = self.token_embedding(src) + self.pos_embedding(src_positions)
        tgt_emb = self.token_embedding(tgt) + self.pos_embedding(tgt_positions)

        # Masks
        src_key_padding_mask = (src == self.pad_id)           # True for pad tokens
        tgt_key_padding_mask = (tgt == self.pad_id)           # True for pad tokens
        tgt_mask = self.make_tgt_mask(tgt)                   # causal mask for decoder

        #Forward through transformer
        out = self.transformer(
            src_emb,
            tgt_emb,
            tgt_mask=tgt_mask,
            src_key_padding_mask=src_key_padding_mask,
            tgt_key_padding_mask=tgt_key_padding_mask
        )

        return self.fc_out(out)

In [ ]:
# Model
model = Seq2SeqTransformer(
    vocab_size=len(vocab),
    pad_id=vocab["<pad>"]
).to(device)

# Optimizer & Loss
optimizer = optim.AdamW(model.parameters(), lr=1e-4)

criterion = nn.CrossEntropyLoss(
    ignore_index=vocab["<pad>"],
    label_smoothing=0.1
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.8,
    patience=10
)

In [ ]:
def load_transformer_checkpoint(checkpoint_path):
    pad_id = vocab["<pad>"]
    vocab_size = len(vocab)

    model = Seq2SeqTransformer(vocab_size, pad_id).to(device)

    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    print(f"Loaded epoch {checkpoint['epoch']} | "
          f"Val Acc: {checkpoint['accuracy']:.4f} | "
          f"Val Loss: {checkpoint['loss']:.4f}")

    return model, checkpoint['epoch']

## Training & Validation Loop (Used for other models as well)

In [ ]:
from tqdm.auto import tqdm
import random

def train_val_model(model, optimizer, criterion, scheduler, epochs, model_type, save_path, load_epochs=0):
    for epoch in range(load_epochs, epochs):
        model.train()
        total_loss = 0
        total_tokens = 0
        correct_tokens = 0

        progress_bar = tqdm(
            train_loader,
            desc=f"Epoch {epoch+1}/{epochs}",
            leave=False
        )

        #Training Phase
        for src, tgt in progress_bar:
            src = src.to(device)
            tgt = tgt.to(device)

            # Skip too-short sequences
            if tgt.size(1) < 2:
                continue

            # teacher forcing: show the model the ground truth in sequence so that it can examine the real previous word to predict the next one
            tgt_input = tgt[:, :-1]
            tgt_output = tgt[:, 1:]

            # Clamp token indices safely
            tgt_input = torch.clamp(tgt_input, 0, len(vocab)-1)
            tgt_output = torch.clamp(tgt_output, 0, len(vocab)-1)

            # Skip batches with invalid token IDs
            if src.max() >= len(vocab) or src.min() < 0:
                print("Invalid src token in batch. Skipping.")
                continue
            if tgt_input.max() >= len(vocab) or tgt_input.min() < 0:
                print("Invalid tgt_input token in batch. Skipping.")
                continue
            if tgt_output.max() >= len(vocab) or tgt_output.min() < 0:
                print("Invalid tgt_output token in batch. Skipping.")
                continue

            optimizer.zero_grad()
            output = model(src, tgt_input)  # (batch, seq_len, vocab_size)

            # Flatten for loss calculation
            output_flat = output.reshape(-1, output.shape[-1])
            tgt_output_flat = tgt_output.reshape(-1)
            loss = criterion(output_flat, tgt_output_flat)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            total_loss += loss.item()

            preds = output.argmax(-1)
            mask = tgt_output != vocab["<pad>"]
            correct_tokens += (preds == tgt_output).masked_select(mask).sum().item()
            total_tokens += mask.sum().item()

            # Update progress bar
            progress_bar.set_postfix(
                batch_loss=loss.item(),
                avg_loss=total_loss / (progress_bar.n + 1),
                batch_acc=correct_tokens / max(1, total_tokens)
            )

        train_loss = total_loss / len(train_loader)
        train_acc = correct_tokens / max(1, total_tokens)
        
        #Validation Phase
        model.eval()
        val_loss = 0
        val_tokens = 0
        val_correct = 0
        with torch.no_grad():
            for src, tgt in val_loader:  # your validation DataLoader
                src = src.to(device)
                tgt = tgt.to(device)

                if tgt.size(1) < 2:
                    continue

                tgt_input = tgt[:, :-1]
                tgt_output = tgt[:, 1:]

                tgt_input = torch.clamp(tgt_input, 0, len(vocab)-1)
                tgt_output = torch.clamp(tgt_output, 0, len(vocab)-1)

                output = model(src, tgt_input)

                output_flat = output.reshape(-1, output.shape[-1])
                tgt_output_flat = tgt_output.reshape(-1)
                loss = criterion(output_flat, tgt_output_flat)
                val_loss += loss.item()

                preds = output.argmax(-1)
                mask = tgt_output != vocab["<pad>"]
                val_correct += (preds == tgt_output).masked_select(mask).sum().item()
                val_tokens += mask.sum().item()

        val_loss /= len(val_loader)
        val_acc = val_correct / max(1, val_tokens)

        scheduler.step(val_loss)
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Current LR: {current_lr:.6f}")

        # Save model for this epoch
        torch.save({
            "epoch": epoch+1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "loss": val_loss,
            "accuracy": val_acc,
            "vocab": vocab
        }, f"{save_path}_epoch{epoch+1}_vAcc{val_acc:.4f}_vLoss{val_loss:.4f}.pth")

        #Sample translation
        model.eval()
        
        if epoch % 5 == 0:
            print("MBR MONITORING")
            # MBR sampling (only a small batch)
            with torch.no_grad():
                sample_batch = next(iter(val_loader))
                idx = random.randint(0, sample_batch[0].size(0)-1)

                sample_src = sample_batch[0][idx].unsqueeze(0).to(device)
                sample_tgt = sample_batch[1][idx]

                pred_tokens = mbr_decode(
                    model,
                    sample_src,
                    max_len=MAX_LEN, #Lower later on
                    sos_id=vocab["<sos>"],
                    eos_id=vocab["<eos>"],
                    model_type=model_type,
                    samples=8
                )

                print("\n--- MBR SAMPLE ---")
                print("SRC :", decode_tokens(sample_src[0].cpu().tolist()))
                print("TGT :", decode_tokens(sample_tgt.tolist()))
                print("PRED:", decode_tokens(pred_tokens))
                print("------------------")

        print(f"Epoch {epoch+1}/{epochs} | "
                f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
                f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

In [ ]:
model, epoch = load_transformer_checkpoint("checkpoints/seq2seq_epoch113_vAcc0.2601_vLoss4.9060.pth")

In [ ]:
train_val_model(model, optimizer, criterion, scheduler, 300, "t", f'checkpoints/seq2seq', epoch)

## Approach 2: LSTM + Attention

In [ ]:
class LSTMEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, pad_id, num_layers=3, dropout=0.3):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True
        )
        self.num_layers = num_layers
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        embedded = self.dropout(self.embedding(src))
        outputs, (hidden, cell) = self.lstm(embedded)

        # Hidden: (num_layers*2, batch, hidden_dim) due to bidirectional
        # Concatenate forward and backward for each layer
        # so decoder gets (num_layers, batch, hidden_dim*2)
        hidden = torch.cat(
            [torch.cat((hidden[2*i], hidden[2*i+1]), dim=1).unsqueeze(0)
             for i in range(self.num_layers)],
            dim=0
        )
        cell = torch.cat(
            [torch.cat((cell[2*i], cell[2*i+1]), dim=1).unsqueeze(0)
             for i in range(self.num_layers)],
            dim=0
        )

        # Now shape becomes:
        # (1, batch, hidden_dim * 2)

        return outputs, hidden, cell

In [ ]:
class LSTMDecoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, pad_id, num_layers=3, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.lstm = nn.LSTM(
            embed_dim + hidden_dim * 2,
            hidden_dim * 2,  # IMPORTANT
            num_layers=num_layers,
            batch_first=True
        )
        self.fc_out = nn.Linear(hidden_dim * 2, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, tgt, hidden, cell, encoder_outputs):
        embedded = self.dropout(self.embedding(tgt))
        batch_size, tgt_len, _ = embedded.shape
        src_len = encoder_outputs.size(1)

        outputs = []

        for t in range(tgt_len):

            # Current decoder hidden state
            hidden_last = hidden[-1]  # (batch, hidden*2)

            # ATTENTION LAYER TO AUGMENT REGULAR LSTM
            attn_scores = torch.bmm(
                encoder_outputs,
                hidden_last.unsqueeze(2)
            ).squeeze(2)  # (batch, src_len)

            attn_weights = torch.softmax(attn_scores, dim=1)

            context = torch.bmm(
                attn_weights.unsqueeze(1),
                encoder_outputs
            )  # (batch, 1, hidden*2)

            # --------------------------------

            lstm_input = torch.cat(
                (embedded[:, t:t+1, :], context),
                dim=2
            )

            output, (hidden, cell) = self.lstm(
                lstm_input,
                (hidden, cell)
            )

            outputs.append(output)

        outputs = torch.cat(outputs, dim=1)
        outputs = self.dropout(outputs)
        
        logits = self.fc_out(outputs)
        return logits, hidden, cell

In [ ]:
class LSTMSeq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, tgt):
        encoder_outputs, hidden, cell = self.encoder(src)
        logits, hidden, cell = self.decoder(tgt, hidden, cell, encoder_outputs)
        return logits

In [ ]:
encoder = LSTMEncoder(len(vocab), 128, 128, vocab["<pad>"], 1)
decoder = LSTMDecoder(len(vocab), 128, 128, vocab["<pad>"], 1)

model = LSTMSeq2Seq(encoder, decoder).to(device)

optimizer = optim.AdamW(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.8,
    patience=10
)

criterion = nn.CrossEntropyLoss(
    ignore_index=vocab["<pad>"],
    label_smoothing=0.1
)

In [ ]:
train_val_model(model, optimizer, criterion, scheduler, 300, f'checkpoints/lstm_seq2seq')

Model Training Ends Here.